# Walkthrough Pipeline ETL - Automobile Dataset

Notebook ini untuk **belajar**: kita jalankan pipeline selangkah demi selangkah, lihat kondisi data di tiap tahap, dan pahami *kenapa* tiap keputusan diambil.

Kode final yang dikumpulkan tetap ada di `src/pipeline.py`. Notebook ini memanggil fungsi-fungsi dari file itu, jadi hasilnya pasti konsisten.

**Alur:** `Extract` (baca CSV) -> `Transform` (inspeksi, cleaning, transformasi) -> `Load` (simpan hasil)

> Cara pakai: jalankan cell dari atas ke bawah (`Shift + Enter`). Notebook ini tidak menimpa file di `data/raw/` maupun `data/processed/`.

## 0. Persiapan

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Cari root project (notebook ada di folder notebooks/) lalu import pipeline.py
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pipeline as pl

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Root project:", ROOT.name)

Root project: data-pipeline-assignment


## 1. Extract - membaca data mentah

Kita baca CSV langsung dengan `pd.read_csv()` ke dalam DataFrame bernama `df`. Ini hanya **membaca** file di `data/raw/`, file mentah tidak pernah ditulis ulang. (Di `pipeline.py`, fungsi `load_data()` melakukan hal yang sama.)

In [2]:
RAW_PATH = ROOT / "data" / "raw" / "automobileEDA_dirty_training.csv"
df = pd.read_csv(RAW_PATH)
df.head()

,transaction_date,symboling,normalized-losses,make,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,width,height,curb-weight,engine-type,num-of-cylinders,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price,city-L/100km,horsepower-binned,diesel,gas
0,2025-01-01,3,122,alfa-romero,std,two,convertible,rwd,front,88.6,0.811148,0.890278,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0,11.190476,Medium,0,1
1,02/01/2025,3,122,alfa-romero,std,two,convertible,rwd,front,88.6,0.811148,0.890278,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0,11.190476,Medium,0,1
2,01-03-2025,1,122,ALFA-ROMERO,std,two,hatchback,rwd,front,94.5,0.822681,0.909722,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0,12.368421,Medium,0,1
3,04-Jan-2025,2,164,audi,std,four,sedan,fwd,front,99.8,0.848630,0.919444,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0,9.791667,Medium,0,1
4,2025-01-05,2,164,audi,std,four,sedan,4wd,front,99.4,0.848630,0.922222,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0,13.055556,Medium,0,1


## 2. Inspeksi - kenali masalahnya dulu

Sebelum membersihkan, kita harus tahu apa yang kotor. Lima hal yang dicek: ukuran, tipe data, missing values, duplikat, dan konsistensi kategori.

In [3]:
print("Ukuran (baris, kolom):", df.shape)
print("Duplicate records    :", df.duplicated().sum())
print()
print("Tipe data kolom transaction_date:", df["transaction_date"].dtype, "(seharusnya datetime)")

Ukuran (baris, kolom): (205, 30)
Duplicate records    : 4

Tipe data kolom transaction_date: str (seharusnya datetime)


In [4]:
# Missing values: tampilkan hanya kolom yang ada kosongnya
missing = df.isna().sum()
missing[missing > 0]

transaction_date     2
make                 2
num-of-doors         2
stroke               4
horsepower           3
price                3
horsepower-binned    1
dtype: int64

Kolom kategorikal sering menyimpan masalah tersembunyi: huruf besar/kecil berbeda dan spasi di ujung teks. Perhatikan tanda kutip pada daftar berikut, spasi di belakang (misalnya `'dodge  '`) jadi terlihat.

In [5]:
for col in ["make", "body-style", "drive-wheels", "fuel-system"]:
    print(col, "->", sorted(df[col].dropna().unique().tolist()))
    print()

make -> ['ALFA-ROMERO', 'Audi', 'BMW', 'alfa-romero', 'audi', 'bmw', 'chevrolet', 'dodge', 'dodge  ', 'honda', 'isuzu', 'jaguar', 'mazda', 'mercedes-benz', 'mercury  ', 'mitsubishi', 'nissan', 'peugot', 'plymouth', 'porsche', 'porsche  ', 'renault', 'saab', 'subaru', 'toyota', 'volkswagen', 'volvo']

body-style -> ['SEDAN', 'Sedan', 'convertible', 'hardtop', 'hatchback', 'sedan', 'wagon']

drive-wheels -> ['4wd', 'AWD', 'RWD', 'fwd', 'rwd']

fuel-system -> ['1bbl', '2bbl', '4bbl', 'MPFI', 'Mpfi', 'idi', 'mfi', 'mpfi', 'spdi', 'spfi']



Kolom `transaction_date` memakai beberapa format sekaligus. Ini masalah karena komputer tidak bisa mengurutkan atau menghitung selisih tanggal dari teks campuran.

In [6]:
df["transaction_date"].dropna().sample(8, random_state=1).tolist()

['2025-06-26',
 '27/06/2025',
 '01-19-2025',
 '2025-03-02',
 '26/05/2025',
 '01-Feb-2025',
 '2025-01-29',
 '11/02/2025']

**Rangkuman temuan:** 17 sel kosong di 7 kolom, 4 duplikat, tanggal bertipe teks dengan 4 format, dan kategori tidak konsisten (`SEDAN`/`Sedan`/`sedan`, `RWD`/`rwd`, `Mpfi`/`mpfi`, spasi berlebih, `peugot`).

> Fungsi `pl.inspect_data(df)` menampilkan semua pemeriksaan ini sekaligus ke terminal.

## 3. Cleaning - membersihkan data

Kita lihat dulu teknik-tekniknya dalam contoh kecil, baru jalankan versi lengkapnya.

### 3a. Seragamkan teks: `strip()` + `lower()`

In [7]:
contoh = pd.Series(["SEDAN", "Sedan", "sedan", " dodge  "])
contoh.str.strip().str.lower().tolist()

['sedan', 'sedan', 'sedan', 'dodge']

### 3b. Seragamkan tanggal

Aturan yang dipakai: tanda `/` dibaca `dd/mm/yyyy`, tanda `-` dengan angka dibaca `mm-dd-yyyy`. Ini asumsi yang bisa diverifikasi karena tanggal di dataset berurutan harian mulai 1 Jan 2025.

In [8]:
for s in ["2025-01-01", "02/01/2025", "01-03-2025", "04-Jan-2025"]:
    print(f"{s:>12}  ->  {pl._parse_date(s).date()}")

  2025-01-01  ->  2025-01-01
  02/01/2025  ->  2025-01-02
  01-03-2025  ->  2025-01-03
 04-Jan-2025  ->  2025-01-04


### 3c. Isi missing values: kenapa median, bukan rata-rata?

Median tidak terpengaruh nilai ekstrem, rata-rata tertarik oleh outlier.

In [9]:
hp = df["horsepower"]
print("Rata-rata horsepower:", round(hp.mean(), 1))
print("Median horsepower   :", hp.median())
print("Nilai maksimum      :", hp.max(), "(outlier yang menarik rata-rata ke atas)")

Rata-rata horsepower: 103.4
Median horsepower   : 95.0
Nilai maksimum      : 262.0 (outlier yang menarik rata-rata ke atas)


Untuk `num-of-doors` (kategorikal) kita pakai **modus per tipe bodi**, karena jumlah pintu sangat ditentukan bentuk mobil:

In [10]:
df.groupby(df["body-style"].str.strip().str.lower())["num-of-doors"].agg(lambda x: x.mode()[0])

body-style
convertible     two
hardtop         two
hatchback       two
sedan          four
wagon          four
Name: num-of-doors, dtype: str

Keputusan lain:
- **`price`, `horsepower`, dan `stroke` diisi median** (10.270, 95, dan 3.29). Median dipilih karena tahan terhadap outlier, misalnya mobil mewah dengan harga 45.400.
- **`horsepower-binned` dihitung ulang** dari `horsepower`, karena kolom ini turunan.
- **`make` kosong diisi `unknown`**, dan **`transaction_date` kosong dipertahankan**, karena keduanya tidak bisa ditebak.

### 3d. Jalankan cleaning lengkap

In [11]:
clean = pl.clean_data(df)


DATA CLEANING
Jumlah baris sebelum cleaning : 205
Jumlah baris sesudah cleaning : 201
Duplicate records dihapus     : 4
Nilai median pengisi          : {'price': 10270.0, 'stroke': 3.29, 'horsepower': 95.0}
Total missing sebelum / sesudah: 17 / 2

Missing values per kolom (yang terdampak):
                    sebelum  sesudah
transaction_date         2        2
make                     2        0
num-of-doors             2        0
stroke                   4        0
horsepower               3        0
price                    3        0
horsepower-binned        1        0

Kolom yang mengalami perubahan: ['transaction_date', 'make', 'num-of-doors', 'body-style', 'drive-wheels', 'fuel-system', 'stroke', 'horsepower', 'price', 'horsepower-binned']


### Cek hasilnya: sebelum vs sesudah

In [12]:
pd.DataFrame({"sebelum": df.isna().sum(), "sesudah": clean.isna().sum()}).query("sebelum > 0 or sesudah > 0")

,sebelum,sesudah
transaction_date,2,2
make,2,0
num-of-doors,2,0
stroke,4,0
horsepower,3,0
price,3,0
horsepower-binned,1,0


In [13]:
print("Baris sebelum -> sesudah:", len(df), "->", len(clean))
print("Tipe transaction_date   :", clean["transaction_date"].dtype)
print()
for col in ["make", "body-style", "drive-wheels", "fuel-system"]:
    print(col, "->", sorted(clean[col].unique().tolist()))
    print()

Baris sebelum -> sesudah: 205 -> 201
Tipe transaction_date   : datetime64[us]

make -> ['alfa-romero', 'audi', 'bmw', 'chevrolet', 'dodge', 'honda', 'isuzu', 'jaguar', 'mazda', 'mercedes-benz', 'mercury', 'mitsubishi', 'nissan', 'peugeot', 'plymouth', 'porsche', 'renault', 'saab', 'subaru', 'toyota', 'unknown', 'volkswagen', 'volvo']

body-style -> ['convertible', 'hardtop', 'hatchback', 'sedan', 'wagon']

drive-wheels -> ['4wd', 'awd', 'fwd', 'rwd']

fuel-system -> ['1bbl', '2bbl', '4bbl', 'idi', 'mfi', 'mpfi', 'spdi', 'spfi']



## 4. Transformation - menyiapkan data untuk analisis/model

Transformasi dijalankan lewat **preprocessing pipeline scikit-learn** (`Pipeline` + `ColumnTransformer`) yang berisi tiga transformer: `MinMaxScaler`, `OrdinalEncoder`, dan `OneHotEncoder`.

### 4a. Kolom mana yang di-scale? (best practice)

Min-Max Scaling hanya untuk **fitur numerik kontinu**. Tidak semua kolom angka perlu di-scale:

In [14]:
print(f"DI-SCALE ({len(pl.SCALE_COLS)} kolom):")
print(pl.SCALE_COLS)

tidak_di_scale = {
    "symboling": "rating risiko ordinal (-2 sampai 3), bukan ukuran kontinu",
    "length": "sudah berskala 0-1 dari sumber dataset",
    "width": "sudah berskala 0-1 dari sumber dataset",
    "diesel": "kolom biner 0/1",
    "gas": "kolom biner 0/1",
}
print("\nTIDAK di-scale:")
for kolom, alasan in tidak_di_scale.items():
    print(f"  {kolom:10} -> {alasan}")

print("\nCatatan: price adalah kolom target, tetapi di tugas ini tetap di-scale (price_scaled)")
print("sebagai latihan. Kolom price asli tetap disimpan. Untuk modeling, target umumnya tidak di-scale.")

DI-SCALE (14 kolom):
['normalized-losses', 'wheel-base', 'height', 'curb-weight', 'engine-size', 'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'city-L/100km', 'price']

TIDAK di-scale:
  symboling  -> rating risiko ordinal (-2 sampai 3), bukan ukuran kontinu
  length     -> sudah berskala 0-1 dari sumber dataset
  width      -> sudah berskala 0-1 dari sumber dataset
  diesel     -> kolom biner 0/1
  gas        -> kolom biner 0/1

Catatan: price adalah kolom target, tetapi di tugas ini tetap di-scale (price_scaled)
sebagai latihan. Kolom price asli tetap disimpan. Untuk modeling, target umumnya tidak di-scale.


### 4b. Cara kerja `MinMaxScaler` (contoh pada satu kolom)

Rumus: `x' = (x - min) / (max - min)`. Nilai terkecil menjadi 0 dan terbesar menjadi 1, sehingga kolom dengan rentang berbeda jauh (misalnya `curb-weight` ribuan dan `bore` sekitar 3) menjadi sebanding.

Alurnya dua langkah: `fit` (scaler mempelajari min dan max) lalu `transform` (menerapkan rumus). `fit_transform` melakukan keduanya sekaligus. Perhatikan data **sebelum** dan **sesudah**:

In [15]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
before = clean[["horsepower"]]            # data SEBELUM scaling
after = scaler.fit_transform(before)      # data SESUDAH scaling (array NumPy)

print("min yang dipelajari scaler:", scaler.data_min_[0])
print("max yang dipelajari scaler:", scaler.data_max_[0])

pd.DataFrame({"horsepower (sebelum)": before["horsepower"], "horsepower (sesudah)": after[:, 0]}).head()

min yang dipelajari scaler: 48.0
max yang dipelajari scaler: 262.0


,horsepower (sebelum),horsepower (sesudah)
0,111.0,0.294393
1,111.0,0.294393
2,154.0,0.495327
3,102.0,0.252336
4,115.0,0.313084


Cek bahwa hasil `MinMaxScaler` sama dengan rumus yang dihitung manual:

In [16]:
x = clean["horsepower"]
manual = (x - x.min()) / (x.max() - x.min())
print("Sama dengan rumus manual:", np.allclose(after[:, 0], manual))

Sama dengan rumus manual: True


### 4c. Encoding

- **Ordinal** untuk kategori yang punya urutan (`Low` < `Medium` < `High`).
- **One-hot** untuk kategori tanpa urutan, tiap nilai jadi kolom 0/1.

In [17]:
pd.get_dummies(clean["aspiration"], prefix="aspiration", dtype=int).head()

,aspiration_std,aspiration_turbo
0,1,0
1,1,0
2,1,0
3,1,0
4,1,0


### 4d. Preprocessing pipeline lengkap

`pl.build_preprocessor()` menggabungkan semua transformer di atas dalam satu objek. Struktur isinya:

In [18]:
features = pl.SCALE_COLS + [pl.ORDINAL_COL] + pl.ONEHOT_COLS
preprocessor = pl.build_preprocessor().fit(clean[features])

for nama, transformer, kolom in preprocessor.named_steps["preprocess"].transformers:
    print(f"{nama:8} {type(transformer).__name__:15} {len(kolom)} kolom")

scale    MinMaxScaler    14 kolom
ordinal  OrdinalEncoder  1 kolom
onehot   OneHotEncoder   3 kolom


Sekarang jalankan transformasi lengkap lewat `pl.transform_data()`, yang memakai pipeline di atas:

In [19]:
final = pl.transform_data(clean)
print()
print("Hasil scaler sama dengan pipeline:", np.allclose(after[:, 0], final["horsepower_scaled"]))
print("Ukuran akhir (baris, kolom)      :", final.shape)


DATA TRANSFORMATION
Kolom di-scale (MinMaxScaler): ['normalized-losses', 'wheel-base', 'height', 'curb-weight', 'engine-size', 'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'city-L/100km', 'price']
Tidak di-scale: symboling, length, width, diesel, gas

Statistik SEBELUM vs SESUDAH Min-Max Scaling:
                   min_sebelum  max_sebelum  min_sesudah  max_sesudah
normalized-losses       65.000      256.000          0.0          1.0
wheel-base              86.600      120.900          0.0          1.0
height                  47.800       59.800          0.0          1.0
curb-weight           1488.000     4066.000          0.0          1.0
engine-size             61.000      326.000          0.0          1.0
bore                     2.540        3.940          0.0          1.0
stroke                   2.070        4.170          0.0          1.0
compression-ratio        7.000       23.000          0.0          1.0
horsepower              

Perbandingan **sebelum** (kolom asli) dan **sesudah** (kolom `_scaled`) Min-Max Scaling. Setelah scaling, min = 0 dan max = 1 untuk semua kolom:

In [20]:
scaled_cols = [c + "_scaled" for c in pl.SCALE_COLS]
pd.DataFrame({
    "min_sebelum": final[pl.SCALE_COLS].min(),
    "max_sebelum": final[pl.SCALE_COLS].max(),
    "min_sesudah": final[scaled_cols].min().set_axis(pl.SCALE_COLS),
    "max_sesudah": final[scaled_cols].max().set_axis(pl.SCALE_COLS),
}).round(3)

,min_sebelum,max_sebelum,min_sesudah,max_sesudah
normalized-losses,65.000,256.000,0.0,1.0
wheel-base,86.600,120.900,0.0,1.0
height,47.800,59.800,0.0,1.0
curb-weight,1488.000,4066.000,0.0,1.0
engine-size,61.000,326.000,0.0,1.0
bore,2.540,3.940,0.0,1.0
stroke,2.070,4.170,0.0,1.0
compression-ratio,7.000,23.000,0.0,1.0
horsepower,48.000,262.000,0.0,1.0
peak-rpm,4150.000,6600.000,0.0,1.0


Perbandingan per baris (kolom asli tetap disimpan di samping kolom hasil transformasi):

In [21]:
final[["horsepower", "horsepower_scaled", "curb-weight", "curb-weight_scaled",
       "engine-size", "engine-size_scaled", "horsepower-binned", "horsepower-binned_encoded"]].head(5)

,horsepower,horsepower_scaled,curb-weight,curb-weight_scaled,engine-size,engine-size_scaled,horsepower-binned,horsepower-binned_encoded
0,111.0,0.294393,2548,0.411171,130,0.260377,Medium,1
1,111.0,0.294393,2548,0.411171,130,0.260377,Medium,1
2,154.0,0.495327,2823,0.517843,152,0.343396,Medium,1
3,102.0,0.252336,2337,0.329325,109,0.181132,Medium,1
4,115.0,0.313084,2824,0.518231,136,0.283019,Medium,1


### 4e. Pipeline bisa dipakai ulang pada data baru

Karena memakai `clip=True` dan `handle_unknown="ignore"`, data baru yang nilainya di luar rentang data latih atau berisi kategori baru tidak merusak hasil:

In [22]:
baru = clean[features].head(2).copy()
baru.loc[baru.index[0], "horsepower"] = 999          # jauh di atas max data latih (262)
baru.loc[baru.index[1], "aspiration"] = "kategori-baru"

hasil = preprocessor.transform(baru)
print("horsepower 999 -> scaled:", float(hasil["scale__horsepower"].iloc[0]), "(tetap maksimal 1.0)")
print("one-hot aspiration untuk kategori baru:")
hasil.filter(like="onehot__aspiration").iloc[[1]]

horsepower 999 -> scaled: 1.0 (tetap maksimal 1.0)
one-hot aspiration untuk kategori baru:


,onehot__aspiration_std,onehot__aspiration_turbo
1,0,0


## 5. Load - hasil akhir

Pipeline menyimpan hasil ke `data/processed/automobileEDA_processed.csv` lewat `pl.save_data()` saat `python src/pipeline.py` dijalankan. Di notebook ini kita cukup **memverifikasi** bahwa hasil di memori sama dengan file yang tersimpan, tanpa menimpanya.

In [23]:
saved = pd.read_csv(pl.PROCESSED_PATH)
print("Ukuran di memori   :", final.shape)
print("Ukuran file tersimpan:", saved.shape)
print("Nama kolom sama    :", list(saved.columns) == list(final.columns))

Ukuran di memori   : (201, 56)
Ukuran file tersimpan: (201, 56)
Nama kolom sama    : True


## Ringkasan

| Tahap | Fungsi | Hasil |
|---|---|---|
| Extract | `load_data()` | 205 baris x 30 kolom |
| Inspeksi | `inspect_data()` | 17 missing, 4 duplikat, tanggal & kategori berantakan |
| Cleaning | `clean_data()` | 201 baris, missing tersisa 2 (hanya `transaction_date`) |
| Transformasi | `transform_data()` | 201 baris x 56 kolom |
| Load | `save_data()` | `data/processed/automobileEDA_processed.csv` |